# Stock Market Prediction with Deep Learning
### LSTM · GRU · Bidirectional LSTM + Attention · Monte Carlo Uncertainty

This notebook builds a complete pipeline for forecasting stock closing prices:

- **Live market data** fetched directly from Yahoo Finance for any ticker
- **20 technical indicator features** — RSI, MACD, Bollinger Bands, ATR, OBV, moving averages and more
- **Three deep learning architectures** trained and compared side by side
- **Monte Carlo Dropout** for honest uncertainty estimation (90% confidence intervals)
- **Five evaluation metrics** — RMSE, MAE, MAPE, R², Directional Accuracy
- **Interactive Plotly charts** for predictions, residuals, and model comparison


## 1 · Install Dependencies

In [1]:
!pip install yfinance plotly -q


## 2 · Imports

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(' All imports successful')


TensorFlow : 2.20.0
NumPy      : 2.0.2
✅ All imports successful


## 3 · Configuration
Change `TICKER` to any Yahoo Finance symbol — AAPL, GOOGL, TSLA, etc.

In [3]:
TICKER       = 'INFY.NS'     # change to any ticker.
                              #    US stocks:    'AAPL', 'MSFT', 'TSLA'
                              #    NSE (India):  'INFY.NS', 'RELIANCE.NS', 'TCS.NS'
                              #    BSE (India):  'INFY.BO', 'RELIANCE.BO'
START_DATE   = '2015-01-01'
END_DATE     = None          # None = today
SEQ_LEN      = 60            # look-back window in trading days
EPOCHS       = 100
BATCH_SIZE   = 32
MC_ITER      = 50            # Monte Carlo dropout passes
TRAIN_RATIO  = 0.80

# Currency is inferred from the ticker suffix -- Yahoo Finance returns NSE/BSE
# prices already denominated in INR, no conversion needed.
if TICKER.upper().endswith(('.NS', '.BO')):
    CURRENCY, CURRENCY_SYM = 'INR', '₹'
else:
    CURRENCY, CURRENCY_SYM = 'USD', '$'

print(f'Ticker: {TICKER}  ->  currency: {CURRENCY} ({CURRENCY_SYM})')


Ticker: INFY.NS  ->  currency: INR (₹)


## 4 · Technical Indicator Engineering

In [4]:
def add_technical_indicators(df):
    close = df['Close']

    # Moving averages
    df['SMA_10']      = close.rolling(10).mean()
    df['SMA_30']      = close.rolling(30).mean()
    df['EMA_12']      = close.ewm(span=12, adjust=False).mean()
    df['EMA_26']      = close.ewm(span=26, adjust=False).mean()

    # MACD
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # Bollinger Bands
    bb_mid            = close.rolling(20).mean()
    bb_std            = close.rolling(20).std()
    df['BB_upper']    = bb_mid + 2 * bb_std
    df['BB_lower']    = bb_mid - 2 * bb_std
    df['BB_width']    = (df['BB_upper'] - df['BB_lower']) / bb_mid

    # RSI (14-day)
    delta             = close.diff()
    gain              = delta.clip(lower=0).rolling(14).mean()
    loss              = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI']         = 100 - (100 / (1 + gain / (loss + 1e-10)))

    # ATR
    hl                = df['High'] - df['Low']
    hpc               = (df['High'] - close.shift(1)).abs()
    lpc               = (df['Low']  - close.shift(1)).abs()
    df['ATR']         = pd.concat([hl, hpc, lpc], axis=1).max(axis=1).rolling(14).mean()

    # On-Balance Volume
    df['OBV']         = (np.sign(close.diff()) * df['Volume']).fillna(0).cumsum()

    # Return & volatility
    df['Return']      = close.pct_change()
    df['Volatility']  = df['Return'].rolling(10).std()

    return df

print(' Technical indicator functions ready')


✅ Technical indicator functions ready


## 5 · Fetch & Prepare Data

In [5]:
def prepare_data(ticker, start, end, seq_len, train_ratio):
    print(f'Downloading {ticker} data...')
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    # Remove any duplicate columns yfinance may produce
    raw = raw.loc[:, ~raw.columns.duplicated()]
    raw.dropna(inplace=True)

    df = add_technical_indicators(raw.copy())
    df.dropna(inplace=True)

    feature_cols = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'SMA_10', 'SMA_30', 'EMA_12', 'EMA_26',
        'MACD', 'MACD_signal', 'MACD_hist',
        'BB_upper', 'BB_lower', 'BB_width',
        'RSI', 'ATR', 'OBV', 'Return', 'Volatility',
    ]
    target_cols = ['High', 'Low', 'Close']   # predict all three price points

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_scaled = scaler_X.fit_transform(df[feature_cols])
    y_scaled = scaler_y.fit_transform(df[target_cols])

    X_seq, y_seq, dates = [], [], []
    for i in range(seq_len, len(df)):
        X_seq.append(X_scaled[i - seq_len: i])
        y_seq.append(y_scaled[i])          # now a 3-value vector, not a scalar
        dates.append(df.index[i])

    X_seq   = np.array(X_seq,  dtype=np.float32)
    y_seq   = np.array(y_seq,  dtype=np.float32)   # shape (N, 3)
    dates   = np.array(dates)

    split        = int(len(X_seq) * train_ratio)
    return {
        'X_train': X_seq[:split],  'X_test': X_seq[split:],
        'y_train': y_seq[:split],  'y_test': y_seq[split:],
        'dates_train': dates[:split], 'dates_test': dates[split:],
        'scaler_X': scaler_X, 'scaler_y': scaler_y, 'raw_df': df,
        'feature_cols': feature_cols, 'target_cols': target_cols,
    }

data         = prepare_data(TICKER, START_DATE, END_DATE, SEQ_LEN, TRAIN_RATIO)
X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
scaler_X     = data['scaler_X']
scaler_y     = data['scaler_y']
target_cols  = data['target_cols']
dates_test   = data['dates_test']
input_shape  = (X_train.shape[1], X_train.shape[2])
y_test_price = scaler_y.inverse_transform(y_test)   # shape (N, 3) -> $ High, Low, Close

print(f'Ticker           : {TICKER}')
print(f'Rows after dropna: {len(data["raw_df"])}')
print(f'Train sequences  : {X_train.shape}')
print(f'Test sequences   : {X_test.shape}')
print(f'Targets          : {target_cols}')


Ticker           : INFY.NS
Rows after dropna: 2818
Train sequences  : (2206, 60, 20)
Test sequences   : (552, 60, 20)
Targets          : ['High', 'Low', 'Close']


## 6 · Feature Correlation with Close Price

In [6]:
# Use only the feature columns we defined — avoids any duplicate Close issue
feat_df = data['raw_df'][data['feature_cols']].copy()
close_series = data['raw_df']['Close'].copy()
if isinstance(close_series, pd.DataFrame):
    close_series = close_series.iloc[:, 0]
feat_df['_Close'] = close_series.values

corr = feat_df.corr()[['_Close']].drop('_Close')
corr.columns = ['Close']
corr = corr.sort_values('Close', ascending=True)

fig = go.Figure(go.Bar(
    x=corr['Close'], y=corr.index, orientation='h',
    marker=dict(color=corr['Close'], colorscale='RdBu', cmid=0),
))
fig.update_layout(
    title=f'{TICKER} — Feature Correlation with Close Price',
    xaxis_title='Pearson r', template='plotly_white', height=550,
)
fig.show()


## 7 · Model Definitions
Three architectures: **Vanilla LSTM**, **GRU**, and **Bidirectional LSTM + Attention**.

In [7]:
# -- Attention Layer --------------------------------------------------------
class BahdanauAttention(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, hidden_states):
        score   = self.V(tf.nn.tanh(self.W(hidden_states)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * hidden_states, axis=1)
        return context, tf.squeeze(weights, -1)


# -- Vanilla LSTM -------------------------------------------------------------
def build_vanilla_lstm(input_shape, n_outputs=3, units=64, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(units, return_sequences=True),
        layers.Dropout(dropout),
        layers.BatchNormalization(),
        layers.LSTM(units // 2),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(n_outputs),
    ], name='VanillaLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='huber', metrics=['mae'])
    return model


# -- GRU ------------------------------------------------------------------------
def build_gru(input_shape, n_outputs=3, units=64, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(units, return_sequences=True),
        layers.Dropout(dropout),
        layers.BatchNormalization(),
        layers.GRU(units // 2),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(n_outputs),
    ], name='GRU')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='huber', metrics=['mae'])
    return model


# -- BiLSTM + Attention -----------------------------------------------------------
def build_bilstm_attention(input_shape, n_outputs=3, units=64, dropout=0.2):
    inp = layers.Input(shape=input_shape)
    x   = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Bidirectional(layers.LSTM(units // 2, return_sequences=True))(x)
    x   = layers.Dropout(dropout)(x)
    ctx, _ = BahdanauAttention(64)(x)
    x   = layers.Dense(64, activation='relu')(ctx)
    x   = layers.Dropout(dropout / 2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(n_outputs)(x)
    model = Model(inputs=inp, outputs=out, name='BiLSTM_Attention')
    model.compile(optimizer=keras.optimizers.Adam(5e-4), loss='huber', metrics=['mae'])
    return model


# -- Shared callbacks ---------------------------------------------------------------
def get_callbacks():
    return [
        callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=7, min_lr=1e-6, verbose=1),
    ]


# -- Monte Carlo Dropout -------------------------------------------------------------
def mc_predict(model, X, n=50):
    preds = np.stack(
        [model(X, training=True).numpy() for _ in range(n)], axis=0
    )
    return preds.mean(0), np.percentile(preds, 5, axis=0), np.percentile(preds, 95, axis=0)


print('Model definitions ready (multi-output: High, Low, Close)')


Model definitions ready (multi-output: High, Low, Close)


## 8 · Train All Three Models


In [8]:
COLORS = {
    'VanillaLSTM':      '#4C72B0',
    'GRU':              '#DD8452',
    'BiLSTM_Attention': '#55A868',
}

model_builders = {
    'VanillaLSTM':      lambda: build_vanilla_lstm(input_shape, n_outputs=len(target_cols)),
    'GRU':              lambda: build_gru(input_shape, n_outputs=len(target_cols)),
    'BiLSTM_Attention': lambda: build_bilstm_attention(input_shape, n_outputs=len(target_cols)),
}

def enforce_hlc_order(arr, target_cols):
    """Reorder independently-predicted High/Low/Close so High >= Close >= Low always holds."""
    idx = {t: i for i, t in enumerate(target_cols)}
    h, l, c = arr[:, idx['High']], arr[:, idx['Low']], arr[:, idx['Close']]
    new_h = np.maximum.reduce([h, l, c])
    new_l = np.minimum.reduce([h, l, c])
    new_c = np.clip(c, new_l, new_h)
    out = arr.copy()
    out[:, idx['High']]  = new_h
    out[:, idx['Low']]   = new_l
    out[:, idx['Close']] = new_c
    return out

trained     = {}
histories   = {}
preds_price = {}   # name -> (N,3) array in $  [High, Low, Close]
mc_bands    = {}   # name -> (mean, lo, hi), each (N,3) in $
metrics     = []   # one row per (Model, Target)

for name, builder in model_builders.items():
    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")
    model = builder()
    hist  = model.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=get_callbacks(),
        verbose=1, shuffle=False,
    )
    trained[name]   = model
    histories[name] = hist.history

    # Point prediction = MC mean, so it's consistent with the CI (both use dropout-on sampling)
    mc_m, mc_lo, mc_hi = mc_predict(model, X_test, MC_ITER)
    y_pred_p = scaler_y.inverse_transform(mc_m)
    mc_lo_p  = scaler_y.inverse_transform(mc_lo)
    mc_hi_p  = scaler_y.inverse_transform(mc_hi)

    # Enforce High >= Close >= Low on both the point prediction and the CI bounds
    y_pred_p = enforce_hlc_order(y_pred_p, target_cols)
    mc_lo_p  = enforce_hlc_order(mc_lo_p, target_cols)
    mc_hi_p  = enforce_hlc_order(mc_hi_p, target_cols)

    preds_price[name] = y_pred_p
    mc_bands[name]     = (y_pred_p, mc_lo_p, mc_hi_p)

    # Metrics, computed separately for each target
    for j, tgt in enumerate(target_cols):
        actual = y_test_price[:, j]
        pred   = y_pred_p[:, j]
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        mape = np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-8))) * 100
        r2   = r2_score(actual, pred)
        da   = (np.sign(np.diff(actual)) == np.sign(np.diff(pred))).mean() * 100
        metrics.append({'Model': name, 'Target': tgt, 'RMSE': rmse, 'MAE': mae,
                        'MAPE (%)': mape, 'R2': r2, 'Dir. Acc. (%)': da})
        print(f'  [{tgt:5s}] RMSE={rmse:.3f}  MAE={mae:.3f}  MAPE={mape:.2f}%  R2={r2:.4f}  DirAcc={da:.1f}%')

print('\nAll models trained on High, Low, and Close')



  VanillaLSTM
Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0161 - mae: 0.1386 - val_loss: 0.0041 - val_mae: 0.0736 - learning_rate: 0.0010
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0338 - mae: 0.1975 - val_loss: 0.0083 - val_mae: 0.1063 - learning_rate: 0.0010
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0232 - mae: 0.1607 - val_loss: 0.0188 - val_mae: 0.1759 - learning_rate: 0.0010
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0174 - mae: 0.1391 - val_loss: 0.0150 - val_mae: 0.1441 - learning_rate: 0.0010
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0080 - mae: 0.0925 - val_loss: 0.0023 - val_mae: 0.0547 - learning_rate: 0.0010
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0055 - mae: 0.0760 - val_loss: 0.0019 - val_mae: 0.0513 - learning_rate: 0.0010
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.0079 - mae: 0.0943 - val_loss: 0.0045 - val_mae: 0.0752 - learning_rate

## 9 · Results Table

In [9]:
results_df = pd.DataFrame(metrics).set_index(['Model', 'Target']).round(4)
print(results_df.to_string())
results_df


                             RMSE       MAE  MAPE (%)      R2  Dir. Acc. (%)
Model            Target                                                     
VanillaLSTM      High    171.2709  120.0615    7.2168  0.3765        49.3648
                 Low     178.9294  123.0350    7.5070  0.3022        47.5499
                 Close   187.2576  129.1688    7.7964  0.2488        48.0944
GRU              High    226.5268  180.4945   11.9270 -0.0906        53.9020
                 Low     247.0305  197.3184   13.2017 -0.3300        51.1797
                 Close   233.6351  190.0103   12.6785 -0.1694        49.5463
BiLSTM_Attention High    237.6216  194.2773   11.9018 -0.2001        54.6279
                 Low     256.2031  215.7021   13.4919 -0.4306        53.1760
                 Close   249.5487  213.7550   13.3160 -0.3342        48.4574


RMSE       MAE  MAPE (%)      R2  Dir. Acc. (%)
Model            Target                                                     
VanillaLSTM      High    171.2709  120.0615    7.2168  0.3765        49.3648
                 Low     178.9294  123.0350    7.5070  0.3022        47.5499
                 Close   187.2576  129.1688    7.7964  0.2488        48.0944
GRU              High    226.5268  180.4945   11.9270 -0.0906        53.9020
                 Low     247.0305  197.3184   13.2017 -0.3300        51.1797
                 Close   233.6351  190.0103   12.6785 -0.1694        49.5463
BiLSTM_Attention High    237.6216  194.2773   11.9018 -0.2001        54.6279
                 Low     256.2031  215.7021   13.4919 -0.4306        53.1760
                 Close   249.5487  213.7550   13.3160 -0.3342        48.4574

## 10 · Training & Validation Loss

In [10]:
fig = go.Figure()
for name, hist in histories.items():
    ep = list(range(1, len(hist['loss']) + 1))
    c  = COLORS[name]
    fig.add_trace(go.Scatter(x=ep, y=hist['loss'],     mode='lines',
                             name=f'{name} train', line=dict(color=c)))
    fig.add_trace(go.Scatter(x=ep, y=hist['val_loss'], mode='lines',
                             name=f'{name} val',   line=dict(color=c, dash='dash')))
fig.update_layout(title='Training & Validation Loss (Huber)',
                  xaxis_title='Epoch', yaxis_title='Loss',
                  template='plotly_white', height=450)
fig.show()


## 11 · Predicted vs Actual Close Price
Shaded bands show the 90% Monte Carlo confidence interval for each model.

In [11]:
fig = go.Figure()
close_idx = target_cols.index('Close')

def hex_to_rgba(hex_color, alpha=0.12):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

fig.add_trace(go.Scatter(x=dates_test, y=y_test_price[:, close_idx], mode='lines',
                         name='Actual Close', line=dict(color='#C44E52', width=2)))

for name, y_pred in preds_price.items():
    c = COLORS[name]
    _, lo, hi = mc_bands[name]
    fig.add_trace(go.Scatter(
        x=list(dates_test) + list(dates_test[::-1]),
        y=list(hi[:, close_idx]) + list(lo[::-1, close_idx]),
        fill='toself', fillcolor=hex_to_rgba(c, 0.12),
        line=dict(color='rgba(0,0,0,0)'),
        name=f'{name} 90% CI', showlegend=True,
    ))
    fig.add_trace(go.Scatter(x=dates_test, y=y_pred[:, close_idx], mode='lines',
                             name=name, line=dict(color=c, width=1.5)))

fig.update_layout(
    title=f'{TICKER} -- Predicted vs Actual Closing Price (Test Set)',
    xaxis_title='Date', yaxis_title=f'Price ({CURRENCY})',
    template='plotly_white', hovermode='x unified', height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()


## 12 · Residuals (Actual − Predicted)

In [12]:
fig = go.Figure()
close_idx = target_cols.index('Close')
for name, y_pred in preds_price.items():
    fig.add_trace(go.Scatter(x=dates_test, y=y_test_price[:, close_idx] - y_pred[:, close_idx],
                             mode='lines', name=name,
                             line=dict(color=COLORS[name])))
fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4)
fig.update_layout(title='Prediction Residuals (Close)',
                  xaxis_title='Date', yaxis_title=f'Residual ({CURRENCY})',
                  template='plotly_white', height=380)
fig.show()


## 13 · Model Comparison Radar Chart

In [13]:
metric_cols = ['RMSE', 'MAE', 'MAPE (%)', 'R2', 'Dir. Acc. (%)']
avg_metrics = pd.DataFrame(metrics).groupby('Model')[metric_cols].mean()
norm = avg_metrics.copy()

# Normalise -- invert error metrics so higher = better everywhere
for col in ['RMSE', 'MAE', 'MAPE (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = 1 - (norm[col] - mn) / (mx - mn + 1e-10)
for col in ['R2', 'Dir. Acc. (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = (norm[col] - mn) / (mx - mn + 1e-10)

fig = go.Figure()
for name in norm.index:
    vals = norm.loc[name].tolist()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=metric_cols + [metric_cols[0]],
        fill='toself', name=name, line=dict(color=COLORS[name]),
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Model Comparison -- averaged across High/Low/Close, higher is better on all axes',
    template='plotly_white', height=480,
)
fig.show()


## 14 · Select the Best Overall Model
Averages RMSE, MAE, MAPE, R², and Directional Accuracy across all three targets (High, Low, Close), normalises them onto the same 0-1 scale, and ranks models by a composite score.


In [14]:
# Average each metric across High / Low / Close for every model
avg_metrics = pd.DataFrame(metrics).groupby('Model')[['RMSE','MAE','MAPE (%)','R2','Dir. Acc. (%)']].mean().round(4)
print('Average performance across High / Low / Close:\n')
print(avg_metrics.to_string())

# Normalise so higher = better on every axis (same logic as the radar chart)
norm = avg_metrics.copy()
for col in ['RMSE', 'MAE', 'MAPE (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = 1 - (norm[col] - mn) / (mx - mn + 1e-10)
for col in ['R2', 'Dir. Acc. (%)']:
    mn, mx = norm[col].min(), norm[col].max()
    norm[col] = (norm[col] - mn) / (mx - mn + 1e-10)

norm['Composite Score'] = norm.mean(axis=1)
norm = norm.sort_values('Composite Score', ascending=False)

best_model_name = norm.index[0]
print(f'\nComposite ranking (0-1 scale, higher = better):\n')
print(norm[['Composite Score']].to_string())

# Count how many individual Target x Metric comparisons each model wins outright (15 total: 3 targets x 5 metrics)
metrics_df = pd.DataFrame(metrics)
wins = {name: 0 for name in avg_metrics.index}
for tgt in target_cols:
    sub = metrics_df[metrics_df['Target'] == tgt].set_index('Model')
    wins[sub['RMSE'].idxmin()] += 1
    wins[sub['MAE'].idxmin()] += 1
    wins[sub['MAPE (%)'].idxmin()] += 1
    wins[sub['R2'].idxmax()] += 1
    wins[sub['Dir. Acc. (%)'].idxmax()] += 1

print(f'\nMetric-wins across all 15 Target x Metric comparisons:')
for name, w in sorted(wins.items(), key=lambda x: -x[1]):
    print(f'  {name}: {w}/15')

print(f'\nBest model: {best_model_name}')
print(f'Justification: {best_model_name} has the highest composite score '
      f"({norm.loc[best_model_name, 'Composite Score']:.3f}) when RMSE, MAE, MAPE, R2, "
      f"and Directional Accuracy are averaged across High, Low, and Close and normalised "
      f"onto a common 0-1 scale, and it wins {wins[best_model_name]}/15 individual "
      f"metric-target comparisons -- the most of the three architectures.")


Average performance across High / Low / Close:

                      RMSE       MAE  MAPE (%)      R2  Dir. Acc. (%)
Model                                                                
BiLSTM_Attention  247.7911  207.9115   12.9032 -0.3216        52.0871
GRU               235.7308  189.2744   12.6024 -0.1967        51.5426
VanillaLSTM       179.1526  124.0885    7.5067  0.3092        48.3364

Composite ranking (0-1 scale, higher = better):

                  Composite Score
Model                            
VanillaLSTM              0.800000
GRU                      0.301323
BiLSTM_Attention         0.200000

Metric-wins across all 15 Target x Metric comparisons:
  VanillaLSTM: 12/15
  BiLSTM_Attention: 2/15
  GRU: 1/15

Best model: VanillaLSTM
Justification: VanillaLSTM has the highest composite score (0.800) when RMSE, MAE, MAPE, R2, and Directional Accuracy are averaged across High, Low, and Close and normalised onto a common 0-1 scale, and it wins 12/15 individual metric-target c

## 15 · Predict the Next Trading Day
Uses the best model (selected above) on the most recent 60-day window to forecast tomorrow's High, Low, and Close, with a 90% Monte Carlo confidence range.


In [15]:
def predict_next_day(model, raw_df, feature_cols, scaler_X, scaler_y, target_cols, seq_len, mc_iter=50):
    last_window = raw_df[feature_cols].values[-seq_len:]
    last_window_scaled = scaler_X.transform(last_window)
    X_input = last_window_scaled.reshape(1, seq_len, len(feature_cols))

    # Monte Carlo dropout: point estimate = mean of the samples, CI = 5th/95th percentile
    mc_stack = np.stack(
        [model(X_input, training=True).numpy()[0] for _ in range(mc_iter)], axis=0
    )
    pred_scaled = mc_stack.mean(axis=0, keepdims=True)
    lo_scaled   = np.percentile(mc_stack, 5, axis=0, keepdims=True)
    hi_scaled   = np.percentile(mc_stack, 95, axis=0, keepdims=True)

    pred_price = scaler_y.inverse_transform(pred_scaled)
    mc_lo      = scaler_y.inverse_transform(lo_scaled)
    mc_hi      = scaler_y.inverse_transform(hi_scaled)

    # Enforce High >= Close >= Low
    pred_price = enforce_hlc_order(pred_price, target_cols)[0]
    mc_lo      = enforce_hlc_order(mc_lo, target_cols)[0]
    mc_hi      = enforce_hlc_order(mc_hi, target_cols)[0]

    return pred_price, mc_lo, mc_hi


# Plausibility guard: flag predictions implying an unrealistic overnight move.
# Most liquid large/mid-cap stocks move single-digit percent on an ordinary day;
# anything beyond this without a known catalyst (earnings, M&A, guidance) should be
# treated as a model reliability issue, not a real signal.
PLAUSIBLE_MOVE_PCT = 10.0

best_model = trained[best_model_name]
pred, lo, hi = predict_next_day(
    best_model, data['raw_df'], data['feature_cols'], scaler_X, scaler_y, target_cols, SEQ_LEN, MC_ITER
)

last_date  = data['raw_df'].index[-1]
next_date  = last_date + pd.tseries.offsets.BDay(1)
last_close = float(data['raw_df']['Close'].iloc[-1])

print(f'Model used  : {best_model_name}  (selected as best performer above)')
print(f'Based on    : last {SEQ_LEN} trading days through {last_date.date()}')
print(f'Last close  : {CURRENCY_SYM}{last_close:.2f}  (reference point for the move below)')
print(f'\nPrediction for {TICKER} on {next_date.date()}:\n')

close_idx = target_cols.index('Close')
for i, tgt in enumerate(target_cols):
    move_pct = (pred[i] - last_close) / last_close * 100
    print(f'  {tgt:6s}: {CURRENCY_SYM}{pred[i]:.2f}   (90% CI: {CURRENCY_SYM}{lo[i]:.2f} - {CURRENCY_SYM}{hi[i]:.2f})   [{move_pct:+.1f}% vs last close]')

close_move_pct = abs((pred[close_idx] - last_close) / last_close * 100)
if close_move_pct > PLAUSIBLE_MOVE_PCT:
    print(f"\n⚠️  WARNING: predicted Close implies a {close_move_pct:.1f}% overnight move "
          f"(threshold: {PLAUSIBLE_MOVE_PCT:.0f}%). This exceeds what's realistic for an ordinary "
          f"trading day without a specific news catalyst (earnings, M&A, guidance change). "
          f"Treat this prediction as unreliable -- it likely reflects model instability or "
          f"insufficient training data rather than a genuine signal. Consider comparing against "
          f"the GRU or Vanilla LSTM model's next-day prediction, or retraining with more history.")
else:
    print(f'\nPredicted move is within a plausible range for an ordinary trading day.')


Model used  : VanillaLSTM  (selected as best performer above)
Based on    : last 60 trading days through 2026-07-10
Last close  : ₹1068.70  (reference point for the move below)

Prediction for INFY.NS on 2026-07-13:

  High  : ₹1112.45   (90% CI: ₹976.18 - ₹1242.54)   [+4.1% vs last close]
  Low   : ₹1088.31   (90% CI: ₹973.65 - ₹1205.00)   [+1.8% vs last close]
  Close : ₹1106.60   (90% CI: ₹974.67 - ₹1242.54)   [+3.5% vs last close]

Predicted move is within a plausible range for an ordinary trading day.
